# Week 3 — CNN + Transformer Image Captioning

**Goal:** Replace the LSTM decoder with a Transformer decoder and compare performance.

### Architecture
```
Image → ResNet50 → Linear(512) ─────────────────────────────────────────────┐
                                                                              ↓
Caption tokens → Embedding(512) + Positional Encoding → Transformer Decoder → FC → Vocab
```

### Key Concepts
- **Self-attention** in decoder (causal mask prevents looking ahead)
- **Cross-attention** attends to image features as memory
- **Pre-LN** (Pre-LayerNorm) for training stability

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torchvision.transforms as T

import config
from src.vocabulary import Vocabulary
from src.dataset import load_captions_df, build_dataloaders
from src.transformer_model import CaptioningTransformer
from src.lstm_model import CaptioningLSTM
from src.train import train
from src.evaluate import compute_bleu

print('Device:', config.DEVICE)

## 1. Setup

In [ ]:
df    = load_captions_df(config.CAPTIONS_FILE)
vocab = Vocabulary.load(os.path.join(config.MODELS_DIR, 'vocab.pkl'))
train_loader, val_loader, test_loader = build_dataloaders(df, vocab, config.IMAGES_DIR)

model = CaptioningTransformer(vocab_size=len(vocab))
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Transformer trainable parameters: {total:,}')

## 2. Attention Mechanism Visualisation (Concept)

In [ ]:
# Visualise the causal (triangular) attention mask used in the decoder
import torch
seq_len = 12
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mask.numpy(), cmap='RdYlGn_r', vmin=0, vmax=1)
ax.set_title('Causal Self-Attention Mask\n(white = masked / cannot attend)', fontsize=11)
ax.set_xlabel('Key position (what we attend TO)')
ax.set_ylabel('Query position (current token)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'attention_mask.png'), dpi=120)
plt.show()

## 3. Positional Encoding Visualisation

In [ ]:
import math
d_model, max_len = 64, 50
pe = torch.zeros(max_len, d_model)
pos = torch.arange(max_len).unsqueeze(1).float()
div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
pe[:, 0::2] = torch.sin(pos * div)
pe[:, 1::2] = torch.cos(pos * div)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pe.numpy().T, aspect='auto', cmap='RdBu')
ax.set_xlabel('Sequence Position')
ax.set_ylabel('Embedding Dimension')
ax.set_title('Sinusoidal Positional Encoding')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'positional_encoding.png'), dpi=120)
plt.show()

## 4. Train Transformer

In [ ]:
history_tf = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    save_name    = 'transformer',
    epochs       = config.NUM_EPOCHS,
    lr           = config.LEARNING_RATE,
    device       = config.DEVICE,
)

## 5. Model Comparison: LSTM vs Transformer

In [ ]:
import pickle
# Load LSTM history if available
# (Run Week 2 notebook first)

fig, ax = plt.subplots(figsize=(10, 5))
epochs = range(1, len(history_tf['val_loss'])+1)
ax.plot(epochs, history_tf['train_loss'], color='#7c6aff', lw=2, label='Transformer Train')
ax.plot(epochs, history_tf['val_loss'],   color='#7c6aff', lw=2, ls='--', label='Transformer Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Transformer Training Curves')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'transformer_loss.png'), dpi=120)
plt.show()

## 6. Beam Search vs Greedy Comparison

In [ ]:
from src.inference import beam_search, load_image

model.eval()
preprocess = T.Compose([
    T.Resize((224,224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

sample = df.drop_duplicates('image').sample(3, random_state=99)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (_, row) in zip(axes, sample.iterrows()):
    img = Image.open(os.path.join(config.IMAGES_DIR, row['image'])).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(config.DEVICE)

    with torch.no_grad():
        greedy = model.caption(tensor, vocab)[0]
        beam   = beam_search(model, tensor, vocab, beam_size=5)

    ax.imshow(img)
    ax.set_title(
        f'Greedy: {greedy}\n\nBeam(5): {beam}\n\nRef: {row["caption"][:50]}',
        fontsize=7
    )
    ax.axis('off')

plt.suptitle('Greedy vs Beam Search (Transformer)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'beam_vs_greedy.png'), dpi=120)
plt.show()

print('Week 3 complete ✓')